# Plan de test : depression_classifier

**Modele** : `depression_classifier.joblib` (ColumnTransformer + LogisticRegression)  
**Dataset** : `student_lifestyle_100k.csv` (100 000 etudiants, cible : Depression 0/1)  
**Composants** : `imputer.joblib`, `scaler.joblib`, `isolation_forest.joblib`

## Cas d'usage

| # | Cas d'usage | Criticite |
|---|---|---|
| CU-1 | Predire la depression a partir du profil d'un etudiant | Haute |
| CU-2 | Detecter les profils atypiques via le score d'anomalie | Moyenne |
| CU-3 | Fonctionner sur des donnees incompletes | Haute |
| CU-4 | Respecter la logique metier (sens des variables sur la cible) | Haute |
| CU-5 | Maintenir les performances sur des sous-populations critiques | Haute |

## Matrice des tests

| ID | Cas d'usage | Scenario | Seuil attendu | Marker |
|---|---|---|---|---|
| S1.1 | CU-1 | Longueur de sortie correcte | `len(y_pred) == n` | `unit` |
| S1.2 | CU-1 | Predictions strictement binaires | `predictions in {0, 1}` | `unit` |
| S1.3 | CU-1 | Tous les artefacts se chargent sans erreur | pas d'exception | `unit` |
| S1.4 | CU-1 | Colonnes de sortie du pipeline presentes | 3 colonnes presentes | `unit` |
| S1.5 | CU-3 | Pipeline sans crash avec des NaN en entree | pas d'exception | `unit` |
| S2.1 | CU-1 | F1-score global | `>= 0.70` | `perf` |
| S2.2 | CU-1 | Recall global sur l'ensemble du dataset | `>= 0.65` | `perf` |
| S3.1 | CU-5 | Recall sur les etudiants reellement depressifs | `>= 0.65` | `slice` |
| S3.2 | CU-4 | Quartile haut de la top feature -> taux depression > moyenne | directionnel | `slice` |
| S3.3 | CU-2 | Recall sur les entrees a faible score d'anomalie | `>= 80 % du recall normal` | `slice` |
| S4.1 | CU-4 | Monotonie : P90 vs P10 de la top feature change les predictions dans le bon sens | directionnel | `business` |
| S5.1 | CU-2 | Taux d'anomalies sur donnees normales | `<= 10 %` | `anomaly` |
| S5.2 | CU-2 | Detection des outliers a +5 ecarts-types | `>= 80 %` | `anomaly` |
| S5.3 | CU-2 | Score d'anomalie dans la plage valide | `in [-1, 1]` | `anomaly` |

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT = Path("atelier-information-et-documentation-main")
df = pd.read_csv(PROJECT / "student_lifestyle_100k.csv")
X  = df.drop(columns=["Depression", "Student_ID"], errors="ignore")
y  = df["Depression"].astype(int)

num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
corr = X[num_cols].assign(Depression=y).corr()["Depression"].drop("Depression")
top_feat = corr.abs().idxmax()

print("Correlations avec Depression :")
print(corr.sort_values(key=abs, ascending=False).to_string())
print(f"\nFeature retenue pour les tests metier et slice : '{top_feat}' ({corr[top_feat]:+.4f})")

---
## 6. CU-5 : Comportement sur sous-populations critiques (slice testing)

**Description** : une bonne metrique globale peut masquer des defaillances sur des sous-groupes. On teste les segments les plus importants pour le cas d'usage.

### Segments identifies

| Segment | Definition | Raison du test |
|---|---|---|
| Depressifs confirmes | `y == 1` | S'assurer que le modele detecte bien les vrais cas |
| Quartile superieur de la top feature | `X[feat] >= Q75` | Cas d'usage principal : etudiants a risque eleve |
| Profils a score d'anomalie bas | `anomaly_score <= P10` | Verifier que les anomalies ne degradent pas le recall |

### Scenarios

| ID | Scenario | Valeur attendue | Marker pytest |
|---|---|---|---|
| S5.1 | Recall sur les etudiants reellement depressifs | `Recall >= 0.65` | `slice` |
| S5.2 | Taux de depression predit sur le quartile superieur de la top feature | superieur a la moyenne globale | `slice` |
| S5.3 | Recall sur les entrees a score d'anomalie bas | `>= 80 % du recall sur les normales` | `slice` |